# A4S 大作业主入口 Notebook

本 notebook 作为项目主程序入口。所有可复用实现都放在 `scripts/` 目录下；这里仅负责导入函数并执行数据集划分流程。

## 1. 创建与 Interformer 对齐的数据划分

本步骤从原始 PDBbind index 和原始结构目录读取数据，再与 `data/splits/interformer` 下的 Interformer 划分 ID 文件进行匹配，生成训练集、验证集和测试集。

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.create_interformer_splits import create_split_table
from scripts.sequence_leakage_check import check_sequence_leakage

RAW_INDEX_PATH = PROJECT_ROOT / "data/raw/pdbbind2020/index/index/INDEX_general_PL.2020R1.lst"
RAW_COMPLEX_ROOT = PROJECT_ROOT / "data/raw/pdbbind2020/complexes/P-L"
INTERFORMER_SPLIT_DIR = PROJECT_ROOT / "data/splits/interformer"

OUTPUT_CSV = PROJECT_ROOT / "data/splits/pdbbind_interformer_splits.csv"
REPORT_JSON = PROJECT_ROOT / "data/splits/pdbbind_interformer_split_report.json"

RAW_INDEX_PATH, RAW_COMPLEX_ROOT, INTERFORMER_SPLIT_DIR

(PosixPath('/Users/songni/canvas.nosync/ai3624/final_project/data/raw/pdbbind2020/index/index/INDEX_general_PL.2020R1.lst'), PosixPath('/Users/songni/canvas.nosync/ai3624/final_project/data/raw/pdbbind2020/complexes/P-L'), PosixPath('/Users/songni/canvas.nosync/ai3624/final_project/data/splits/interformer'))

In [2]:
split_df, split_report = create_split_table(
    index_path=RAW_INDEX_PATH,
    complex_root=RAW_COMPLEX_ROOT,
    split_dir=INTERFORMER_SPLIT_DIR,
)

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
split_df.to_csv(OUTPUT_CSV, index=False)
REPORT_JSON.write_text(json.dumps(split_report, indent=2, ensure_ascii=False), encoding="utf-8")

OUTPUT_CSV, REPORT_JSON

(PosixPath('/Users/songni/canvas.nosync/ai3624/final_project/data/splits/pdbbind_interformer_splits.csv'), PosixPath('/Users/songni/canvas.nosync/ai3624/final_project/data/splits/pdbbind_interformer_split_report.json'))

## 2. 划分结果汇总

In [3]:
split_df["split"].value_counts().rename_axis("split").reset_index(name="count")

,split,count
0,train,16038
1,unassigned,1698
2,valid,945
3,test,356


In [4]:
split_df.groupby("split")["pAffinity"].agg(["count", "mean", "min", "max"]).reset_index()

,split,count,mean,min,max
0,test,353,6.548434,0.657577,11.301030
1,train,15947,6.412734,0.397940,15.221849
2,unassigned,1685,6.260795,0.995679,12.301030
3,valid,938,6.405778,0.602060,11.397940


In [5]:
summary_keys = [
    "raw_rows",
    "primary_split_file_counts",
    "matched_primary_split_counts",
    "unassigned_raw_rows",
    "primary_split_overlap_counts",
    "split_ids_missing_from_raw_counts",
    "raw_ids_not_in_primary_interformer_splits_count",
    "eval_subset_counts_in_raw",
]
{key: split_report[key] for key in summary_keys}

{'raw_rows': 19037, 'primary_split_file_counts': {'train': 16379, 'valid': 968, 'test': 363}, 'matched_primary_split_counts': {'train': 16038, 'valid': 945, 'test': 356}, 'unassigned_raw_rows': 1698, 'primary_split_overlap_counts': {'test_vs_train': 0, 'test_vs_valid': 0, 'train_vs_valid': 0}, 'split_ids_missing_from_raw_counts': {'train': 341, 'valid': 23, 'test': 7}, 'raw_ids_not_in_primary_interformer_splits_count': 1698, 'eval_subset_counts_in_raw': {'is_coreset': 285, 'is_diff_test_core': 641, 'is_posebusters_pdb_ccd': 0, 'is_test_no_rec_overlap': 141, 'is_test_sanitizable': 326}}

## 3. 蛋白序列相似性泄露检查

题目要求训练集、验证集和测试集之间的蛋白序列相似性低于阈值，例如 40%。含义是：验证集或测试集中的蛋白不应与训练集蛋白过于相似，否则模型可能只是记住了训练集中相近蛋白家族的模式，而不是真正泛化到新蛋白。这里用 MMseqs2 检查 valid/test 蛋白链与 train 蛋白链之间是否存在序列 identity 不低于 40%、覆盖度不低于 80% 的命中。

In [6]:
SEQUENCE_LEAKAGE_DIR = PROJECT_ROOT / "data/splits/sequence_leakage"

sequence_leakage_hits, sequence_leakage_report = check_sequence_leakage(
    split_csv=OUTPUT_CSV,
    output_dir=SEQUENCE_LEAKAGE_DIR,
    min_seq_id=0.4,
    coverage=0.8,
    min_length=30,
)

sequence_leakage_report

{'split_csv': '/Users/songni/canvas.nosync/ai3624/final_project/data/splits/pdbbind_interformer_splits.csv', 'method': 'mmseqs easy-search', 'min_seq_id': 0.4, 'coverage': 0.8, 'min_length': 30, 'chain_sequence_rows': 34094, 'chain_sequence_counts_by_split': {'test': 701, 'train': 31460, 'valid': 1933}, 'complex_counts_by_split': {'test': 356, 'train': 16036, 'valid': 945}, 'hit_count': 207985, 'leakage_detected': True, 'sequence_table_path': '/Users/songni/canvas.nosync/ai3624/final_project/data/splits/sequence_leakage/chain_sequences.csv', 'hits_path': '/Users/songni/canvas.nosync/ai3624/final_project/data/splits/sequence_leakage/sequence_leakage_hits.csv', 'fasta_paths': {'train': '/Users/songni/canvas.nosync/ai3624/final_project/data/splits/sequence_leakage/train_chains.fasta', 'valid_test': '/Users/songni/canvas.nosync/ai3624/final_project/data/splits/sequence_leakage/valid_test_chains.fasta'}}

In [7]:
sequence_leakage_hits.head(20)

,query,target,pident,alnlen,qlen,tlen,evalue,bits,query_split,query_pdb_id,query_chain_id,target_split,target_pdb_id,target_chain_id
0,valid|1p06|A,train|1p05|A,100.0,198,198,198,1.188000e-122,385,valid,1p06,A,train,1p05,A
1,valid|1p06|A,train|9lpr|A,100.0,198,198,198,1.188000e-122,385,valid,1p06,A,train,9lpr,A
2,valid|1p06|A,train|1p04|A,100.0,198,198,198,1.188000e-122,385,valid,1p06,A,train,1p04,A
3,valid|1p06|A,train|1p03|A,100.0,198,198,198,1.188000e-122,385,valid,1p06,A,train,1p03,A
4,valid|1p06|A,train|1p02|A,100.0,198,198,198,1.188000e-122,385,valid,1p06,A,train,1p02,A
5,valid|1p06|A,train|2h5d|A,100.0,198,198,198,1.188000e-122,385,valid,1p06,A,train,2h5d,A
6,valid|1p06|A,train|1p01|A,100.0,198,198,198,1.188000e-122,385,valid,1p06,A,train,1p01,A
7,valid|1p06|A,train|1p10|A,99.4,198,198,198,7.883000e-122,382,valid,1p06,A,train,1p10,A
8,valid|1p06|A,train|8lpr|A,99.4,198,198,198,7.883000e-122,382,valid,1p06,A,train,8lpr,A
9,valid|1p06|A,train|5lpr|A,99.4,198,198,198,7.883000e-122,382,valid,1p06,A,train,5lpr,A


## 4. 数据预览

In [8]:
split_df.head(10)

,split,split_source,line_number,pdb_id,resolution,release_year,raw_affinity,reference,ligand_name,comment,raw_line,index_parse_ok,affinity_type,affinity_relation,affinity_value,affinity_unit,affinity_molar,pAffinity,affinity_parse_ok,resolution_numeric,complex_dir,year_bucket,protein_path,has_protein,protein_bytes,pocket_path,has_pocket,pocket_bytes,ligand_sdf_path,has_ligand_sdf,ligand_sdf_bytes,ligand_mol2_path,has_ligand_mol2,ligand_mol2_bytes,is_coreset,is_diff_test_core,is_posebusters_pdb_ccd,is_test_no_rec_overlap,is_test_sanitizable
0,train,Interformer timesplit/no-lig-overlap files,7,2tpi,2.10,1982,Kd=49uM,2tpi.pdf,2-mer,,2tpi 2.10 1982 Kd=49uM // 2tpi.pdf (2...,True,Kd,=,49.000,uM,4.900000e-05,4.309804,True,2.1,/Users/songni/canvas.nosync/ai3624/final_proje...,1981-2000,/Users/songni/canvas.nosync/ai3624/final_proje...,True,343278,/Users/songni/canvas.nosync/ai3624/final_proje...,True,87237,/Users/songni/canvas.nosync/ai3624/final_proje...,True,2766,/Users/songni/canvas.nosync/ai3624/final_proje...,True,3837,False,False,False,False,False
1,train,Interformer timesplit/no-lig-overlap files,8,5tln,2.30,1982,Ki=0.43uM,5tln.pdf,BAN,incomplete ligand structure,5tln 2.30 1982 Ki=0.43uM // 5tln.pdf (B...,True,Ki,=,0.430,uM,4.300000e-07,6.366532,True,2.3,/Users/songni/canvas.nosync/ai3624/final_proje...,1981-2000,/Users/songni/canvas.nosync/ai3624/final_proje...,True,395361,/Users/songni/canvas.nosync/ai3624/final_proje...,True,101574,/Users/songni/canvas.nosync/ai3624/final_proje...,True,3211,/Users/songni/canvas.nosync/ai3624/final_proje...,True,4444,False,False,False,False,False
2,train,Interformer timesplit/no-lig-overlap files,9,4tln,2.30,1982,Ki=190uM,4tln.pdf,LNO,,4tln 2.30 1982 Ki=190uM // 4tln.pdf (LNO),True,Ki,=,190.000,uM,1.900000e-04,3.721246,True,2.3,/Users/songni/canvas.nosync/ai3624/final_proje...,1981-2000,/Users/songni/canvas.nosync/ai3624/final_proje...,True,395685,/Users/songni/canvas.nosync/ai3624/final_proje...,True,84402,/Users/songni/canvas.nosync/ai3624/final_proje...,True,1843,/Users/songni/canvas.nosync/ai3624/final_proje...,True,2574,False,False,False,False,False
3,train,Interformer timesplit/no-lig-overlap files,10,4cts,2.90,1984,Kd<10uM,4cts.pdf,OAA,,4cts 2.90 1984 Kd<10uM // 4cts.pdf (OAA),True,Kd,<,10.000,uM,1.000000e-05,5.000000,True,2.9,/Users/songni/canvas.nosync/ai3624/final_proje...,1981-2000,/Users/songni/canvas.nosync/ai3624/final_proje...,True,1128978,/Users/songni/canvas.nosync/ai3624/final_proje...,True,93312,/Users/songni/canvas.nosync/ai3624/final_proje...,True,849,/Users/songni/canvas.nosync/ai3624/final_proje...,True,1219,False,False,False,False,False
4,train,Interformer timesplit/no-lig-overlap files,11,6rsa,NMR,1986,Ki=40uM,6rsa.pdf,UVC,,6rsa NMR 1986 Ki=40uM // 6rsa.pdf (UVC),True,Ki,=,40.000,uM,4.000000e-05,4.397940,True,NaN,/Users/songni/canvas.nosync/ai3624/final_proje...,1981-2000,/Users/songni/canvas.nosync/ai3624/final_proje...,True,179658,/Users/songni/canvas.nosync/ai3624/final_proje...,True,81810,/Users/songni/canvas.nosync/ai3624/final_proje...,True,2586,/Users/songni/canvas.nosync/ai3624/final_proje...,True,3605,False,False,False,False,False
5,train,Interformer timesplit/no-lig-overlap files,12,1rnt,1.90,1987,Kd=6.5uM,1rnt.pdf,2GP,,1rnt 1.90 1987 Kd=6.5uM // 1rnt.pdf (2GP),True,Kd,=,6.500,uM,6.500000e-06,5.187087,True,1.9,/Users/songni/canvas.nosync/ai3624/final_proje...,1981-2000,/Users/songni/canvas.nosync/ai3624/final_proje...,True,127089,/Users/songni/canvas.nosync/ai3624/final_proje...,True,57348,/Users/songni/canvas.nosync/ai3624/final_proje...,True,2752,/Users/songni/canvas.nosync/ai3624/final_proje...,True,3801,False,False,False,False,False
6,train,Interformer timesplit/no-lig-overlap files,13,6cha,1.80,1987,Ki=40uM,6cha.pdf,PBA,covalent complex,6cha 1.80 1987 Ki=40uM // 6cha.pdf (P...,True,Ki,=,40.000,uM,4.000000e-05,4.397940,True,1.8,/Users/songni/canvas.nosync/ai3624/final_proje...,1981-2000,/Users/songni/canvas.nosync/ai3624/final_proje...,True,581985,/User